In [1]:
import os
import random
import numpy as np
import pandas as pd

# Reproducibility (score-neutral; avoids run-to-run volatility)
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Resolve correct Kaggle input root
# (Your provided paths show /kaggle/input/...; keep fallback for older ../input usage)
INPUT_ROOT_CANDIDATES = [
    "/kaggle/input/jigsaw-toxic-comment-classification-challenge",
    "../input/jigsaw-toxic-comment-classification-challenge",
    "/kaggle/data/jigsaw-toxic-comment-classification-challenge",
    "../input",
]
INPUT_ROOT = None
for p in INPUT_ROOT_CANDIDATES:
    if os.path.exists(p):
        # If pointing at ../input, ensure competition subfolder exists
        if os.path.basename(p) == "input":
            comp = os.path.join(p, "jigsaw-toxic-comment-classification-challenge")
            if os.path.exists(comp):
                INPUT_ROOT = comp
                break
        else:
            INPUT_ROOT = p
            break

if INPUT_ROOT is None:
    raise FileNotFoundError("Could not locate Kaggle input data directory.")

print("Using INPUT_ROOT:", INPUT_ROOT)
print("Files:", sorted([f for f in os.listdir(INPUT_ROOT) if f.endswith(".csv")])[:10])



Using INPUT_ROOT: /kaggle/input/jigsaw-toxic-comment-classification-challenge
Files: ['sample_submission.csv', 'test.csv', 'train.csv']


In [2]:
# Use tensorflow.keras to avoid Keras/protobuf incompatibilities in Kaggle runtimes
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    GlobalMaxPool1D,
    Dropout,
    Embedding,
    Bidirectional,
    Flatten,
    Conv1D,
    MaxPool1D,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tqdm.auto import tqdm

# Make TF deterministic-ish (best effort; score-neutral)
try:
    tf.random.set_seed(SEED)
except Exception:
    pass



2026-05-12 04:49:57.404569: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 04:49:57.415926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778561397.428811      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778561397.432264      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 04:49:57.446209: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
train_path = os.path.join(INPUT_ROOT, "train.csv")
test_path = os.path.join(INPUT_ROOT, "test.csv")
sample_path = os.path.join(INPUT_ROOT, "sample_submission.csv")

df = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print(df.shape, test.shape, sample_submission.shape)
df.head()



(159571, 8) (153164, 2) (153164, 7)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [4]:
list_classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y = df[list_classes].values.astype("float32")

# Ensure text is string and NaNs handled
x = df["comment_text"].fillna("").astype(str)
x_test = test["comment_text"].fillna("").astype(str)



In [5]:
# Tokenize (core logic preserved: num_words=20000 and maxlen=50)
max_features = 20000
maxlen = 50
embed_size = 50

token = Tokenizer(num_words=max_features)
token.fit_on_texts(x)

seq = token.texts_to_sequences(x)
pad_seq = pad_sequences(seq, maxlen=maxlen)

test_seq = token.texts_to_sequences(x_test)
test_pad_seq = pad_sequences(test_seq, maxlen=maxlen)

vocab_size = min(max_features, len(token.word_index) + 1)
print("vocab_size:", vocab_size)
print("Train padded shape:", pad_seq.shape, "Test padded shape:", test_pad_seq.shape)



vocab_size: 20000
Train padded shape: (159571, 50) Test padded shape: (153164, 50)


In [6]:
# The original notebook tried to load external GloVe from ../input/gloveembeddings/... which is not available.
# Minimal fix: keep identical embedding layer interface but use a seeded random embedding matrix.
# This preserves core model architecture while removing the missing-file hard dependency.
emb_mean, emb_std = 0.0, 1.0
embedding_matrix = np.random.normal(emb_mean, emb_std, (vocab_size, embed_size)).astype(
    "float32"
)



In [7]:
# Model 1: Bidirectional LSTM + GlobalMaxPool
# Replace CuDNNLSTM with LSTM for CPU/GPU compatibility (same semantics for units/return_sequences).
model1 = Sequential()
model1.add(
    Embedding(
        vocab_size,
        embed_size,
        input_length=maxlen,
        weights=[embedding_matrix],
        trainable=False,
    )
)
model1.add(Bidirectional(LSTM(50, return_sequences=True)))
model1.add(GlobalMaxPool1D())
model1.add(Dense(50, activation="relu"))
model1.add(Dropout(0.2))
model1.add(Dense(6, activation="sigmoid"))
model1.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history1 = model1.fit(
    pad_seq, y, epochs=2, batch_size=32, validation_split=0.1, verbose=2
)
model1.summary()



Epoch 1/2


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-12 04:50:26.494763: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


4488/4488 - 47s - 10ms/step - accuracy: 0.9283 - loss: 0.0899 - val_accuracy: 0.9940 - val_loss: 0.0705
Epoch 2/2
4488/4488 - 45s - 10ms/step - accuracy: 0.9940 - loss: 0.0652 - val_accuracy: 0.9940 - val_loss: 0.0671


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 50)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 100)        │        40,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 100)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           306 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,137,270 (4.34 MB)

 Trainable params: 45,756 (178.73 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

 Optimizer params: 91,514 (357.48 KB)

In [8]:
predict1 = model1.predict(test_pad_seq, batch_size=1024, verbose=1)
sub1 = sample_submission.copy()
sub1[list_classes] = predict1
sub1.to_csv("submission1.csv", index=False)
print("Wrote submission1.csv", sub1.shape)



150/150 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step
Wrote submission1.csv (153164, 7)


In [9]:
# Model 2: single LSTM
model2 = Sequential()
model2.add(
    Embedding(
        vocab_size,
        embed_size,
        input_length=maxlen,
        weights=[embedding_matrix],
        trainable=False,
    )
)
model2.add(LSTM(50))
model2.add(Dense(50, activation="relu"))
model2.add(Dense(6, activation="sigmoid"))
model2.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history2 = model2.fit(
    pad_seq, y, epochs=2, batch_size=32, validation_split=0.1, verbose=2
)

predict2 = model2.predict(test_pad_seq, batch_size=1024, verbose=1)
sub2 = sample_submission.copy()
sub2[list_classes] = predict2
sub2.to_csv("submission2.csv", index=False)
print("Wrote submission2.csv", sub2.shape)



Epoch 1/2
4488/4488 - 36s - 8ms/step - accuracy: 0.9825 - loss: 0.0926 - val_accuracy: 0.9939 - val_loss: 0.0740
Epoch 2/2
4488/4488 - 33s - 7ms/step - accuracy: 0.9942 - loss: 0.0683 - val_accuracy: 0.9940 - val_loss: 0.0676
150/150 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step
Wrote submission2.csv (153164, 7)


In [10]:
# Model 3: Conv1D (original used Convolution1D; Conv1D is the modern alias)
model3 = Sequential()
model3.add(
    Embedding(
        vocab_size,
        embed_size,
        input_length=maxlen,
        weights=[embedding_matrix],
        trainable=False,
    )
)
model3.add(Conv1D(9, kernel_size=5, activation="relu"))
model3.add(MaxPool1D(2))
model3.add(Flatten())
model3.add(Dense(50, activation="relu"))
model3.add(Dense(6, activation="sigmoid"))
model3.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history3 = model3.fit(
    pad_seq, y, epochs=2, batch_size=32, validation_split=0.1, verbose=2
)

predict3 = model3.predict(test_pad_seq, batch_size=1024, verbose=1)
sub3 = sample_submission.copy()
sub3[list_classes] = predict3
sub3.to_csv("submission3.csv", index=False)
print("Wrote submission3.csv", sub3.shape)



Epoch 1/2
4488/4488 - 8s - 2ms/step - accuracy: 0.9187 - loss: 0.1174 - val_accuracy: 0.9928 - val_loss: 0.1056
Epoch 2/2
4488/4488 - 7s - 2ms/step - accuracy: 0.9896 - loss: 0.1007 - val_accuracy: 0.9939 - val_loss: 0.1023
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Wrote submission3.csv (153164, 7)


In [11]:
# Model 4: Conv1D with different filter/kernel
model4 = Sequential()
model4.add(
    Embedding(
        vocab_size,
        embed_size,
        input_length=maxlen,
        weights=[embedding_matrix],
        trainable=False,
    )
)
model4.add(Conv1D(18, kernel_size=3, activation="relu"))
model4.add(MaxPool1D(2))
model4.add(Flatten())
model4.add(Dense(50, activation="relu"))
model4.add(Dense(6, activation="sigmoid"))
model4.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history4 = model4.fit(
    pad_seq, y, epochs=2, batch_size=32, validation_split=0.1, verbose=2
)

predict4 = model4.predict(test_pad_seq, batch_size=1024, verbose=1)
sub4 = sample_submission.copy()
sub4[list_classes] = predict4
sub4.to_csv("submission4.csv", index=False)
print("Wrote submission4.csv", sub4.shape)



Epoch 1/2
4488/4488 - 9s - 2ms/step - accuracy: 0.9528 - loss: 0.1099 - val_accuracy: 0.9925 - val_loss: 0.0969
Epoch 2/2
4488/4488 - 8s - 2ms/step - accuracy: 0.9901 - loss: 0.0906 - val_accuracy: 0.9930 - val_loss: 0.0917
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Wrote submission4.csv (153164, 7)


In [12]:
# Ensemble (core logic preserved: simple average of 4 model predictions)
ensemble_prediction = (
    0.25 * predict1 + 0.25 * predict2 + 0.25 * predict3 + 0.25 * predict4
)

submission = sample_submission.copy()
submission[list_classes] = ensemble_prediction

# Safety: clip to valid probability range (score-neutral; protects against any numerical spillover)
submission[list_classes] = submission[list_classes].clip(0.0, 1.0)

# Ensure correct column order
submission = submission[["id"] + list_classes]

submission.to_csv("submission_ensemble.csv", index=False)
print("Wrote submission_ensemble.csv", submission.shape)
print(submission.head())

Wrote submission_ensemble.csv (153164, 7)
                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.664452      0.165851  0.526741  0.041535  0.472897   
1  0000247867823ef7  0.035581      0.000527  0.008560  0.000191  0.006610   
2  00013b17ad220c46  0.053610      0.001818  0.019056  0.000696  0.015624   
3  00017563c3f7919a  0.010992      0.000083  0.002094  0.000029  0.002089   
4  00017695ad8997eb  0.185797      0.009440  0.099512  0.001428  0.066987   

   identity_hate  
0       0.104157  
1       0.001278  
2       0.004573  
3       0.000186  
4       0.009889  
